---


### **作業上繳設定**

1. **檔案 → 在雲端硬碟中儲存副本**
2. 對老師分享的 `2026檢定培訓營` 按 **「新增雲端硬碟捷徑」** 加到「我的雲端硬碟」（課堂會引導）
3. 確認老師給的是 **編輯者** 權限（否則無法上傳）
4. 填好下方 **DAY**（Day1、Day2…）與 **STUDENT_NAME**（你的姓名）
5. 依序執行 3 個灰色程式格：設定 → 掛載 Drive → **上繳工具**（應顯示 `上繳工具已載入（2026-06-15）`）
6. 每題流程：**寫程式 → 執行作答格 → 執行 `make_submit_button("Prob_K")` 格 → 按上繳**

上繳後檔案會出現在：

```
2026檢定培訓營 / Day1 / 王小明 / Prob_K.py
```

> `Day1`、姓名等子資料夾**不用事先建立**，第一次上繳會自動產生。
> 若改過答案，要**再執行一次** `make_submit_button(...)` 那格，再按上繳。


In [ ]:
# 老師分享的共用資料夾（捷徑到「我的雲端硬碟」後的路徑，通常不用改）
SUBMIT_BASE = "/content/drive/MyDrive/橘子蘋果/APCS/2026檢定培訓營"

# ===== 請填入 =====
DAY = "Day1"           # 統一格式：Day1、Day2、Day3...
STUDENT_NAME = "王小明"  # 你的姓名


In [ ]:
#@title 首次使用需授權 Google 帳號
from google.colab import drive
drive.mount("/content/drive")
print("Google Drive 已掛載")


In [ ]:
#@title 上傳相關設定
import os

import ipywidgets as widgets
from IPython.display import clear_output, display
from IPython import get_ipython

_NOTEBOOK_CACHE = None
SUBMIT_TOOL_VERSION = "2026-06-15"


def _user_ns():
    ip = get_ipython()
    return ip.user_ns if ip is not None else globals()


def _resolve_submit_folder():
    ns = _user_ns()
    base = ns.get("SUBMIT_BASE", "")
    day = ns.get("DAY", "").strip()
    name = ns.get("STUDENT_NAME", "").strip()

    if not base:
        raise ValueError("SUBMIT_BASE 未設定")
    if not os.path.isdir(base):
        raise ValueError(
            f"找不到共用資料夾：{base}\n"
            "請確認已掛載 Drive，且已將「2026檢定培訓營」捷徑加到「我的雲端硬碟」"
        )
    if not day:
        raise ValueError("請填 DAY（例如 Day1）")
    if not day.startswith("Day") or not day[3:].isdigit():
        raise ValueError("DAY 請用 Day1、Day2 格式")
    if not name:
        raise ValueError("請填 STUDENT_NAME（你的姓名）")

    return os.path.join(base, day, name)


def _submit_filename(question_id):
    return f"{question_id}.py"


def _cell_source_text(cell):
    src = cell.get("source", [])
    if isinstance(src, str):
        return src
    return "".join(src)


def _capture_notebook_now():
    """在執行 cell 當下讀取 Colab 畫面上的講義（不能在按鈕 callback 裡讀）。"""
    try:
        from google.colab import _message

        reply = _message.blocking_request("get_ipynb", request="", timeout_sec=10)
        if reply and "ipynb" in reply:
            return reply["ipynb"]
    except Exception:
        pass
    return None


def _extract_answer_code(nb, question_id):
    marker = f"#start-{question_id}"
    for cell in nb.get("cells", []):
        if cell.get("cell_type") != "code":
            continue
        src = _cell_source_text(cell)
        if marker not in src:
            continue
        lines = []
        for line in src.splitlines():
            stripped = line.strip()
            if stripped == marker or stripped == "#start":
                continue
            lines.append(line)
        code = "\n".join(lines).strip()
        return code + "\n" if code else ""
    raise ValueError(
        f"找不到作答區 {marker}。請確認開啟的是「_上繳」版講義，並在作答格寫好程式。"
    )


def make_submit_button(question_id):
    global _NOTEBOOK_CACHE

    nb = _capture_notebook_now()
    if nb is None:
        print("⚠️ 無法讀取講義。請在 Google Colab 中執行此格。")
        _NOTEBOOK_CACHE = None
    else:
        _NOTEBOOK_CACHE = nb
        try:
            _extract_answer_code(nb, question_id)
        except ValueError as exc:
            print(f"⚠️ {exc}")

    btn = widgets.Button(description="上繳", button_style="success", icon="cloud-upload")
    output = widgets.Output()

    def on_submit(_):
        with output:
            clear_output(wait=True)
            print("上繳中，請稍候…")
            try:
                if _NOTEBOOK_CACHE is None:
                    print("⚠️ 請先重新執行此格（make_submit_button），再按上繳。")
                    return

                if not os.path.isdir("/content/drive/MyDrive"):
                    print("⚠️ 請先執行上方的「掛載 Google Drive」cell")
                    return

                code = _extract_answer_code(_NOTEBOOK_CACHE, question_id)
                if not code.strip():
                    print("⚠️ 作答區是空的，請先寫程式，再重新執行此格，然後按上繳")
                    return

                target_dir = _resolve_submit_folder()
                os.makedirs(target_dir, exist_ok=True)

                filename = _submit_filename(question_id)
                filepath = os.path.join(target_dir, filename)

                with open(filepath, "w", encoding="utf-8") as f:
                    f.write(code)

                print("✅ 上繳成功！")
                print(f"檔案：{filename}")
                print(f"路徑：{filepath}")
            except Exception as exc:
                print(f"❌ 上繳失敗：{exc}")

    btn.on_click(on_submit)
    display(widgets.VBox([btn, output]))


print(f"上繳工具已載入（{SUBMIT_TOOL_VERSION}）")

# 商業類術科精選｜配合 Lesson 4

對應 **Lesson4**：`字串應用(2).ipynb`、`函式與模組(1).ipynb`（字串走訪、查表、模組化邏輯）。

題組來自術科精選（112–113，Lesson3 範圍），依本週教材主題分配，**不重複**。

**本檔收錄：**
- **112 學年度 Problem K**
- **112 學年度 Problem J**
- **113 學年度 Problem H**
- **113 學年度 Problem I**


## Problem K 字串

### 題目（PDF 擷取）

```text
Problem Description
有一個長度為n的字串s，由小寫字母構成。想要從s 移去兩個連續的字母，然後拼接起來，
對於所有的連續兩個字母的移除，能產生的新的字串有多少種。
例如，有一個字串“aaabcc”。您可以得到以下不同的字串：「abcc」（刪除前兩個”aa”或
第二個和第三個字元”aa”），「aacc」（刪除第三個和第四個字元”ab”），「aaac」（刪除
第四個和第五個字元”bc”）和「aaab」（刪除最後兩個”cc”）。
Input Format
輸入資料的第一列包含一個整數𝑡(1 ≤𝑡≤103) —測試案例的數量。
每組測試資料有二列，描述如下：
每組測試資料描述的第一列包含一個整數𝑛(3 ≤𝑛≤2 × 106)。
每組測試資料描述的第二列包含一個長度為𝑛的字串𝑠，由小寫字母組成。
Output Format
對於每組測試資料，列印一個整數，透過刪除兩個連續字母可以獲得的不同字串的數量。
Sample Input 1
7
6
aaabcc
10
aaaaaaaaaa
6
abcdef
7
abacaba
6
cccfff
4
abba
5
ababa
Sample Output 1
4
1
5
3
3
3
1
Sample Input 2
2
69
tqwkcomrhawhlizimbvajzjbuifleqpnkbgondlwubfwsifgtdwxipnhrakdqywjxjemb
Sample Output 2
65
67
13

71
nhlisuruyheyckzarqkuxsrnpivuowgoryvhtnwazbwuwjoabumgmuqibwmdoagpubfimbj
Hint
在Sample Input 1中，在第三個範例中”abcdef”，取得以下字串：「cdef」、「adef」、
「abef」、「abcf」、「abcd」。
在第七個範例中”ababa”，任何刪除都會產生字串「aba」。
```

### 參考解答


**（註：原檔標為會超時的解法）**

In [ ]:
#start

**（註：原檔標為較易讀的解法）**

In [ ]:
#start-Prob_K


In [ ]:
make_submit_button("Prob_K")


---

## Problem J 七段顯示器

### 題目（PDF 擷取）

```text
Problem Description
一個七段顯示器，可以顯示一個數字，如圖，我們可以列出，顯示「0」需要6段，顯示
「1」需要2段，往後依序是5段、5段、4段、5段、6段、3段、7段、6段。片段數量一定時，
我們可以透過選擇相同片段中較大的數字（例如5段可以顯示“2”、“3”、“5”，選擇
“5”為最佳），還可以透過將一定量的片段拆分組成更多位的數字（例如4段可以顯示
“4”，但我們可以拆分成兩個2段來顯示“11”）。兩種方法一比較，方法很顯然是，有條
件的情況下盡可能選擇更多位數字，否則則取一定量片段中最大的數字。
分別需要一定數量的片段（如圖，顯示「1」需要兩個片段）。問提供𝑛個片段時，表示可
以亮的段數，問能顯示出來最大的值是多少。
七段顯示器：打開2段，可以顯示1。打開3段，可以顯示7。打開4段，可以顯示4。打開5段，
可以顯示2、3、5、6。打開6段，可以顯示0、9。打開7段，可以顯示8。題目要求顯示最大
數字，因此每一位數要打開盡量少段的顯示器。每打開2 段顯示器，就可以多一個位數（並
顯示1 在該位數）。需留下2 或3（預先判斷𝑛mod 2 為偶數或奇數），用來顯示最高位數1
或7。
Input Format
輸入資料的第一列包含一個整數𝑡(1 ≤𝑡≤103) —測試案例的數量。然後是測試案例，每個
測試案例都由單獨的一列表示，其中包含一個整數𝑛(2 ≤𝑛≤106)－相應測試案例中可以開
啟的最大段數。
Output Format
對於每個測試案例，列印透過開啟不超過𝑛段螢幕可以顯示的最大整數。請注意，答案可能
不適合標準32 位元或64 位元整數資料類型。
Sample Input 1
2
3
4
Sample Output 1
7
11
Hint
其中最小的兩個分別是2段的“1”和3段的“7”，顯然這兩個數字是“最經濟的”，我們只
要這兩個數字即可。選擇方法便是當片段數量是偶數時，全吧用來顯示“1”；當片段數是
奇數時，留有三段用來顯示“7”，其餘全顯示“1”。
```

### 參考解答


In [ ]:
#start-Prob_J


In [ ]:
make_submit_button("Prob_J")


---

## Problem H 身分證檢驗

### 題目（PDF 擷取）

```text
Problem Description
身分證字號有底下這樣的規則：
(1) 英文代號用下表轉換成數字
A=10 台北市, J=18 新竹縣, S=26 高雄縣
B=11 台中市, K=19 苗栗縣, T=27 屏東縣
C=12 基隆市, L=20 台中縣, U=28 花蓮縣
D=13 台南市, M=21 南投縣, V=29 台東縣
E=14 高雄市, N=22 彰化縣, W=32 金門縣
F=15 台北縣, O=35 新竹市, X=30 澎湖縣
G=16 宜蘭縣, P=23 雲林縣, Y=31 陽明山
H=17 桃園縣, Q=24 嘉義縣, Z=33 連江縣
I=34 嘉義市, R=25 台南縣,
(2) 英文轉成的數字, 個位數乘9再加上十位數的數字
(3) 各數字從右到左依次乘1, 2, 3, 4, . . . , 8，並且相加
(4) 求出(2),(3) 及最後一碼的和
(5) 如果求出的和可以被10整除，則為合法身分證字號輸出Y，否則為非法身分證字號輸出N
例如：T112663836
2 + 7 × 9 + 1 × 8 + 1 × 7 + 2 × 6 + 6 × 5 + 6 × 4 + 3 × 3 + 8 × 2 + 3 × 1 + 6 = 180
因為180可以被10 整除，此身分證字號為合法身分證字號Y。
例如：A123456789
1 + 0 × 9 + 1 × 8 + 2 × 7 + 3 × 6 + 4 × 5 + 5 × 4 + 6 × 3 + 7 × 2 + 8 × 1 + 9 = 130
第一個數字為1，代表是男生，若為2代表女生。不用檢查第一個數字是否為1 或2，也不用檢
查身分證字號格式。
Input Format
輸入共一行。每一行包含一組身分證字號。
Output Format
每讀入一行身分證字號，輸出Y 或N 。
Sample Input 1
T112663836
Sample Output 1
Y
Sample Input 2
S154287863
Sample Output 2
N
8
```

### 參考解答


In [ ]:
#start-Prob_H


In [ ]:
make_submit_button("Prob_H")


---

## Problem I 密碼強度檢測

### 題目（PDF 擷取）

```text
Problem Description
密碼通常包含大小寫字母、數字和特殊字符。給定一個密碼（字串），請撰寫一程式，每項
合格得1 分，總分為5 分，檢查密碼規則如下：
1. 密碼長度至少要有八個字元(含)以上；
2. 至少要有一個數字；
3. 至少要有一個大寫英文字母；
4. 至少要有一個小寫英文字母；
5. 至少要有一個特殊字符[ ! @ # $ ˆ & * ( ) , . ? ” :
{ } | <
> ]
Input Format
一個密碼字符串（長度不超過100）。
Output Format
密碼強度評級，請輸出該密碼的評分1 ∼5。
Sample Input 1
P@ssw0rd123!
Sample Output 1
5
Sample Input 2
Password
Sample Output 2
3
Sample Input 3
word
Sample Output 3
1
Sample Input 4
password
Sample Output 4
2
Sample Input 5
PASS
Sample Output 5
1
Sample Input 6
123456
Sample Output 6
1
Sample Input 7
@@@ ###
Sample Output 7
1
9
```

### 參考解答


In [ ]:
#start-Prob_I


In [ ]:
make_submit_button("Prob_I")
